# Chapter 6 &mdash; Watching It Fail: Parity, and What Hedging Looks Like

**Concept 15 of the Chapter 6 decomposition:** *Watching It Fail: Parity, and What Hedging Looks Like*

Trained on parity it agrees on exactly half the windows &mdash; and reports the same probability for all of them, because it truly cannot tell them apart.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-Watching-It-Fail-Parity/Concept-Watching-It-Fail-Parity.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.GPTLab         import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The previous concept predicts that no context-$k$ model can learn parity. Here it is,
trained anyway.

It agrees with the two-state minimal DFA on **four of eight** windows &mdash; chance
&mdash; and the interesting part is *how* it fails. It does not guess wildly. It
reports very nearly the **same** $P(\text{END})$, about $0.2$, for every window.

That flatness is the signature of indistinguishability. The model has averaged over
contexts it cannot separate, which is the best any estimator can do when the evidence
really is identical. Concept 6's *indistinguishability* is not an abstraction here; it
is a number that comes out the same eight times.

*Needs `torch`, which is preinstalled on Colab.*

## 2. Definitions

### Parity, and the prediction

In [ ]:
# --- the languages we will ask about ------------------------------------
ENDS01 = md2mc('''DFA
I  : 0 -> S0
I  : 1 -> I
S0 : 0 -> S0
S0 : 1 -> F
F  : 0 -> S0
F  : 1 -> I
''')

PARITY = md2mc('''DFA
IF : 0 -> IF
IF : 1 -> S1
S1 : 0 -> S1
S1 : 1 -> IF
''')

DIV3 = md2mc('''DFA
IF : 0 -> IF
IF : 1 -> S1
S1 : 0 -> S2
S1 : 1 -> IF
S2 : 0 -> S1
S2 : 1 -> S2
''')

NO11 = md2mc('''DFA
IF : 0 -> IF
IF : 1 -> F1
F1 : 0 -> IF
F1 : 1 -> D
D  : 0|1 -> D
''')

from jove.GPTLab import *

print('minimal DFA:', len(min_dfa(PARITY)['Q']), 'states')
print('k_local at k = 1..6:', [k_local(PARITY, k)[0] for k in range(1, 7)])
print()
print('Predicted before training: chance agreement, at every k.')

<!-- nav-strip -->

---

&larr;&nbsp;[Ch6&nbsp;14.&nbsp;Which DFAs a $k$-Window Can Learn: Myhill-Nerode Against a Window](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-Which-DFAs-A-Window-Can-Learn/Concept-Which-DFAs-A-Window-Can-Learn.ipynb) &nbsp;&middot;&nbsp; [**Chapter 6** index](https://github.com/ganeshutah/Jove/blob/master/Chapter6-DFAOps/README.md) &nbsp;&middot;&nbsp; [Ch7&nbsp;1.&nbsp;Nondeterminism as Forking Tokens, and as Guessing](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7-NFA/Concept-Forking-Tokens/Concept-Forking-Tokens.ipynb)&nbsp;&rarr;

---

## 3. Tests

**Train it anyway.**

In [ ]:
ok, why = torch_available()
acc, seq = corpus(PARITY)
print('accepted strings:', len(acc), ' corpus tokens:', len(seq))
if ok:
    g = train(seq, k=3, iters=500)
    print('trained on %d examples, final loss %.4f' % (g.examples, g.final_loss))
else:
    print('torch is not available here:', why)

**The result**, and the shape of the failure.

In [ ]:
if ok:
    sep, margin, rows = report(g, PARITY, k=3, seen=windows_seen(seq, 3))
    ends = [p[2] for _, p, _ in rows]
    print()
    print('P(END) ranges over %.2f to %.2f -- a spread of %.2f'
          % (min(ends), max(ends), max(ends) - min(ends)))
    print('Eight windows, one answer.  It is not confused; it is averaging.')
    assert not sep
else:
    print('(on Colab: not separated, every P(END) between 0.18 and 0.23.)')

**Compare with the language it can learn.** Same model, same corpus size, same training.

In [ ]:
if ok:
    seq2 = corpus(ENDS01)[1]
    g2 = train(seq2, k=3)
    sep2, m2, rows2 = report(g2, ENDS01, k=3, show=False,
                             seen=windows_seen(seq2, 3))
    ends2 = [p[2] for _, p, _ in rows2]
    print('%-13s %-11s %-20s %s'
          % ('language', 'separated?', 'P(END) spread', 'k-local?'))
    for nm, sp, e, D in (('parity', sep, ends, PARITY),
                         ('ends in 01', sep2, ends2, ENDS01)):
        print('%-13s %-11s %-20s %s'
              % (nm, sp, '%.2f to %.2f' % (min(e), max(e)), k_local(D, 3)[0]))
    print()
    print('A wide spread means the windows told it something.  A flat one')
    print('means they could not -- and the k_local column says, without')
    print('training anything, which it was going to be.')
else:
    print('(on Colab: parity not separated, spread 0.05; ends-in-01')
    print(' separated, spread 0.63.)')

**The two strings at the bottom of it.**

In [ ]:
a, b, tail = colliding_prefixes(PARITY, 3)
print('%r and %r both end in %r' % (a, b, tail))
print('  ones in %-6r : %d  -> accepted %s' % (a, a.count('1'), accepts_dfa(PARITY, a)))
print('  ones in %-6r : %d  -> accepted %s' % (b, b.count('1'), accepts_dfa(PARITY, b)))
print()
print('After either one the model is in the same state, because its state')
print('IS the last three symbols.  The language wants opposite answers.')
print()
print('A fixed context window is a finite memory, and no amount of data')
print('buys memory the architecture does not have.')
assert accepts_dfa(PARITY, a) != accepts_dfa(PARITY, b)

## 4. Exercises


1. Raise $k$ to 5 and retrain. Does agreement improve? The previous concept says what
   to expect; check that it is right.
2. Train far longer &mdash; `iters=5000`. What happens to the loss, and what happens
   to the spread of $P(\text{END})$?
3. Parity **is** regular and a two-state machine decides it. Explain to someone who
   says "so the model just needs more data" exactly what is wrong with that.
4. Give the model a token it could use as memory &mdash; for instance train on strings
   with a running parity bit interleaved. Does it learn now? What have you actually
   changed?
5. Chapter 11 asks this question of context-free languages. Before reading it: is a
   transformer with unbounded context a pushdown automaton? What would you need to
   check?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter6-DFAOps/Concept-Watching-It-Fail-Parity')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')